In [33]:

# ============================================================
# NOTEBOOK 10 — RECOGNITION WITH AUGMENTED DATA
# Real + traditional augmentation, ResNet18, train/evaluate
# ============================================================

from pathlib import Path
import json, time, random, copy, csv
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm

# ---------------- Configuration ----------------
SEED = 42
BATCH_SIZE = 32
EPOCHS = 8
LR = 1e-3
NUM_WORKERS = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT = Path("/kaggle/working/ubiris_v2_project")
PRE_DIR = PROJECT / "notebook02_preprocessing"
BASELINE_DIR = PROJECT / "notebook04_baseline_recognition"
TRAD_DIR = PROJECT / "notebook05_traditional_augmentation"
MANIFEST_DIR = PROJECT / "notebook09_dataset_augmentation"
OUT_DIR = PROJECT / "notebook10_augmented_recognition"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_DIR = PRE_DIR / "images_224"
TRAIN_CSV = BASELINE_DIR / "UBIRIS_V2_train_split.csv"
VAL_CSV = BASELINE_DIR / "UBIRIS_V2_validation_split.csv"
TEST_CSV = BASELINE_DIR / "UBIRIS_V2_test_split.csv"
TRAD_DIR_IMAGES = TRAD_DIR / "augmented_train_images"
MANIFEST_CSV = MANIFEST_DIR / "UBIRIS_V2_augmented_training_manifest.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

for p in [IMG_DIR, TRAIN_CSV, VAL_CSV, TEST_CSV, TRAD_DIR_IMAGES, MANIFEST_CSV]:
    assert p.exists(), f"Missing required path: {p}"

# ---------------- Load splits and manifest ----------------
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)
manifest = pd.read_csv(MANIFEST_CSV)

# Use only actual real and traditional images in the manifest.
train_manifest = manifest[
    manifest["source_type"].isin(["real", "traditional_augmented"])
].copy()

assert len(train_manifest) > 0
assert not train_manifest["split"].isin(["validation", "test"]).any()

print("\nTraining manifest rows:", len(train_manifest))
print("Original validation images:", len(val_df))
print("Original test images:", len(test_df))
print("\nTraining source counts:")
print(train_manifest["source_type"].value_counts().to_string())

# Verify every manifest path exists.
missing = [p for p in train_manifest["path"].astype(str) if not Path(p).is_file()]
assert len(missing) == 0, f"Missing training files: {len(missing)}"

# ---------------- Dataset ----------------
class ManifestDataset(Dataset):
    def __init__(self, df, is_manifest=False):
        self.df = df.reset_index(drop=True)
        self.is_manifest = is_manifest
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.is_manifest:
            path = Path(str(row["path"]))
            label = int(row["label"])
        else:
            path = IMG_DIR / (Path(str(row["filename"])).stem + ".png")
            label = int(row["label"])

        with Image.open(path) as im:
            image = self.transform(im.convert("L"))
        return image, label

train_dataset = ManifestDataset(train_manifest, is_manifest=True)
val_dataset = ManifestDataset(val_df)
test_dataset = ManifestDataset(test_df)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
    drop_last=False
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)

# Infer number of classes from all split labels.
num_classes = int(max(
    train_df["label"].max(),
    val_df["label"].max(),
    test_df["label"].max()
)) + 1

print("\nClasses:", num_classes)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# ---------------- ResNet18 ----------------
model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(
    1, 64, kernel_size=7, stride=2, padding=3, bias=False
)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

def unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

def evaluate(loader):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    all_labels, all_preds = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(
                device_type="cuda", dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                logits = model(images)
                loss = criterion(logits, labels)

            preds = logits.argmax(dim=1)
            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())

    return {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "labels": all_labels,
        "predictions": all_preds
    }

# ---------------- Train ----------------
HISTORY_PATH = OUT_DIR / "UBIRIS_V2_augmented_recognition_history.csv"
CHECKPOINT_PATH = OUT_DIR / "UBIRIS_V2_augmented_resnet18_best.pth"
history = []
best_val_acc = -1.0
best_state = None
start_all = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_seen = 0
    correct = 0
    epoch_start = time.time()

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type="cuda", dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss in epoch {epoch}")

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * labels.size(0)
        total_seen += labels.size(0)
        correct += (preds == labels).sum().item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{correct / max(total_seen, 1):.3f}"
        )

    train_loss = total_loss / max(total_seen, 1)
    train_acc = correct / max(total_seen, 1)
    val_result = evaluate(val_loader)
    epoch_seconds = time.time() - epoch_start

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "validation_loss": val_result["loss"],
        "validation_accuracy": val_result["accuracy"],
        "epoch_seconds": round(epoch_seconds, 2),
        "learning_rate": LR
    }
    history.append(record)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    if val_result["accuracy"] > best_val_acc:
        best_val_acc = val_result["accuracy"]
        best_state = copy.deepcopy(unwrap(model).state_dict())
        torch.save({
            "epoch": epoch,
            "model_state_dict": best_state,
            "num_classes": num_classes,
            "validation_accuracy": best_val_acc,
            "training_sources": ["real", "traditional_augmented"]
        }, CHECKPOINT_PATH)

    print(
        f"\nEpoch {epoch}: train_loss={train_loss:.4f}, "
        f"train_acc={train_acc:.4%}, "
        f"val_loss={val_result['loss']:.4f}, "
        f"val_acc={val_result['accuracy']:.4%}, "
        f"time={epoch_seconds / 60:.2f} min"
    )

# ---------------- Best checkpoint test evaluation ----------------
assert best_state is not None
unwrap(model).load_state_dict(best_state)
test_result = evaluate(test_loader)

try:
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_result["labels"], test_result["predictions"],
        average="macro", zero_division=0
    )
except Exception:
    precision = recall = f1 = None

test_metrics = {
    "test_accuracy": test_result["accuracy"],
    "test_loss": test_result["loss"],
    "macro_precision": precision,
    "macro_recall": recall,
    "macro_f1": f1,
    "test_images": len(test_dataset),
    "training_images": len(train_dataset),
    "best_validation_accuracy": best_val_acc,
    "best_checkpoint_epoch": int(pd.DataFrame(history).loc[
        pd.DataFrame(history)["validation_accuracy"].idxmax(), "epoch"
    ]),
    "epochs_trained": EPOCHS,
    "note": (
        "Training uses real plus traditional augmented images. "
        "GAN and diffusion individual images were unavailable. "
        "Original split is image-level, not subject-disjoint; labels are filename-derived."
    )
}

METRICS_PATH = OUT_DIR / "UBIRIS_V2_augmented_recognition_test_metrics.json"
METRICS_CSV = OUT_DIR / "UBIRIS_V2_augmented_recognition_test_metrics.csv"
METRICS_PATH.write_text(json.dumps(test_metrics, indent=2))
pd.DataFrame([test_metrics]).to_csv(METRICS_CSV, index=False)

print("\nFINAL TEST METRICS")
print(json.dumps(test_metrics, indent=2))
print(f"\nTotal training duration: {(time.time() - start_all) / 60:.2f} min")
print("Checkpoint:", CHECKPOINT_PATH)
print("History:", HISTORY_PATH)
print("Metrics:", METRICS_PATH)
print("Metrics CSV:", METRICS_CSV)


PyTorch: 2.11.0+cu128
Device: cuda
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Training manifest rows: 12729
Original validation images: 778
Original test images: 778

Training source counts:
source_type
traditional_augmented    8486
real                     4243

Classes: 260
Train batches: 398
Validation batches: 25
Test batches: 25


Epoch 1/8:   0%|          | 0/398 [00:00<?, ?it/s]


Epoch 1: train_loss=5.0344, train_acc=3.0639%, val_loss=4.4193, val_acc=5.0129%, time=0.78 min


Epoch 2/8:   0%|          | 0/398 [00:00<?, ?it/s]


Epoch 2: train_loss=3.3880, train_acc=19.6795%, val_loss=2.2594, val_acc=39.4602%, time=0.76 min


...


Epoch 6: train_loss=0.1565, train_acc=95.9698%, val_loss=0.3509, val_acc=90.7455%, time=0.76 min


Epoch 7/8:   0%|          | 0/398 [00:00<?, ?it/s]


Epoch 7: train_loss=0.1172, train_acc=96.9754%, val_loss=0.1472, val_acc=96.2725%, time=0.76 min


Epoch 8/8:   0%|          | 0/398 [00:00<?, ?it/s]


Epoch 8: train_loss=0.1035, train_acc=97.2504%, val_loss=0.4458, val_acc=85.7326%, time=0.77 min

FINAL TEST METRICS
{
  "test_accuracy": 0.9331619537275064,
  "test_loss": 0.21380237793516377,
  "macro_precision": 0.9533608058608057,
  "macro_recall": 0.9392307692307692,
  "macro_f1": 0.9351090576090577,
  "test_images": 778,
  "training_images": 12729,
  "best_validation_accuracy": 0.9627249357326478,
  "best_checkpoint_epoch": 7,
  "epochs_trained": 8,
  "note": "Training uses real plus traditional augmented images. GAN and diffusion individual images were unavailable. Original split is image-level, not subject-disjoint; labels are filename-derived."
}

Total training duration: 6.17 min
Checkpoint: /kaggle/working/ubiris_v2_project/notebook10_augmented_recognition/UBIRIS_V2_augmented_resnet18_best.pth
History: /kaggle/working/ubiris_v2_project/notebook10_augmented_recognition/UBIRIS_V2_augmented_recognition_history.csv
Metrics: /kaggle/working/ubiris_v2_project/notebook10_augmented

## Conclusion

A ResNet18-based ocular recognition model was trained using a combined training dataset containing 4,243 original training images and 8,486 traditionally augmented images, for a total of 12,729 training samples. The original validation and test sets were kept unchanged, containing 778 images each.

The model was trained for eight epochs using two Tesla T4 GPUs. The best validation accuracy was **96.27% at epoch 7**. Evaluation on the test set produced **93.32% accuracy, 95.34% macro precision, 93.92% macro recall, and 93.51% macro F1-score**.

Compared with the baseline experiment, the augmented-data experiment achieved substantially higher test metrics. These findings indicate that the combined real and traditionally augmented training dataset was effective in this experimental setup. However, a controlled experiment is required to isolate the contribution of augmentation from other training-run differences.

The final epoch showed a decrease in validation accuracy, suggesting possible overfitting or validation instability. The best-validation checkpoint was retained for evaluation.

**Limitations:** The dataset split was image-level rather than subject-disjoint, and class labels were derived from filenames. Therefore, the results do not demonstrate generalization to unseen identities. Individual GAN- and diffusion-generated images were unavailable for this experiment, so their contribution to recognition performance remains unevaluated.

Overall, this experiment establishes a strong augmented-data recognition baseline for subsequent comparisons and privacy-related evaluation.

In [34]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
import json

PROJECT = Path("/kaggle/working/ubiris_v2_project")
OUT_DIR = PROJECT / "notebook10_augmented_recognition"

required_files = [
    "UBIRIS_V2_augmented_resnet18_best.pth",
    "UBIRIS_V2_augmented_recognition_history.csv",
    "UBIRIS_V2_augmented_recognition_test_metrics.json",
    "UBIRIS_V2_augmented_recognition_test_metrics.csv",
]

print("NOTEBOOK 10 — OUTPUT VERIFICATION")
missing = []

for filename in required_files:
    path = OUT_DIR / filename
    if path.exists() and path.stat().st_size > 0:
        print(f"OK   {filename} ({path.stat().st_size:,} bytes)")
    else:
        print(f"MISS {filename}")
        missing.append(filename)

if missing:
    raise FileNotFoundError(
        "Missing or empty outputs: " + ", ".join(missing)
    )

metrics_path = OUT_DIR / "UBIRIS_V2_augmented_recognition_test_metrics.json"
with open(metrics_path, "r") as f:
    metrics = json.load(f)

print("\nFINAL METRICS CHECK")
print(f"Test accuracy: {metrics['test_accuracy'] * 100:.2f}%")
print(f"Macro F1:      {metrics['macro_f1'] * 100:.2f}%")
print(f"Best val acc:  {metrics['best_validation_accuracy'] * 100:.2f}%")
print(f"Best epoch:    {metrics['best_checkpoint_epoch']}")

zip_path = PROJECT / "UBIRIS_V2_Notebook10_Results.zip"

with ZipFile(zip_path, "w", ZIP_DEFLATED) as zf:
    for path in sorted(OUT_DIR.rglob("*")):
        if path.is_file():
            zf.write(path, path.relative_to(PROJECT))

print("\nZIP CREATED")
print(f"Path: {zip_path}")
print(f"Size: {zip_path.stat().st_size / (1024 * 1024):.2f} MB")
display(FileLink(str(zip_path)))


NOTEBOOK 10 — OUTPUT VERIFICATION
OK   UBIRIS_V2_augmented_resnet18_best.pth (45,297,291 bytes)
OK   UBIRIS_V2_augmented_recognition_history.csv (822 bytes)
OK   UBIRIS_V2_augmented_recognition_test_metrics.json (546 bytes)
OK   UBIRIS_V2_augmented_recognition_test_metrics.csv (478 bytes)

FINAL METRICS CHECK
Test accuracy: 93.32%
Macro F1:      93.51%
Best val acc:  96.27%
Best epoch:    7

ZIP CREATED
Path: /kaggle/working/ubiris_v2_project/UBIRIS_V2_Notebook10_Results.zip
Size: 40.02 MB


/kaggle/working/ubiris_v2_project/UBIRIS_V2_Notebook10_Results.zip